In [2]:
# for hotpotqa and 2wikimqa
import string
import re

def is_number_with_commas(s):
    pattern = re.compile(r'^\d{1,3}(,\d{3})*$')
    return bool(pattern.match(s))

def normalize_answer(s):
  def remove_articles(text):
    return re.sub(r"\b(a|an|the)\b", " ", text)
  
  def white_space_fix(text):
      return " ".join(text.split())

  def remove_punc(text):
      exclude = set(string.punctuation)
      return "".join(ch for ch in text if ch not in exclude)

  def lower(text):
      return text.lower()

  return white_space_fix(remove_articles(remove_punc(lower(s))))


def exact_match_score(predictions, references):
    cnt = 0
    for pred, ref in zip(predictions, references):
        # print(ref[0])
        gt = normalize_answer(ref)
        pred = [normalize_answer(p) for p in pred]
        for p in pred:
            if(gt in p):
                # print('Groud:'+gt+'Predict:'+p)
                cnt += 1
                break
    if len(predictions) == 0:
        return 0
    return cnt / len(predictions)

# 示例
import pandas as pd
import json
result_file = 'YOUR_PARSE_RESULT_FILE.csv' # your parse result file after running the parse_result.py
df1 = pd.read_csv(result_file)
answer = df1['Final Answer'].values.tolist()
new_answer = []
for idx in range(len(answer)):
    a = str(answer[idx])
    a = a.strip("{}[]").strip("'")
    if is_number_with_commas(a):
        a_list = [a]
    else:
        a_list = a.split(",")
    for i in range(len(a_list)):
        a_list[i] = a_list[i].strip("'").strip().strip('"')
    new_answer.append(a_list)

raw_qa_json = '../fine-tuning/data/hotpotqa_500.json' # '../fine-tuning/data/hotpotqa_500.json' | '../fine-tuning/data/2wikmqa_500.json'
answer = []
raw_qa = json.load(open(raw_qa_json,'r'))
for item in raw_qa:
    answer.append(item['answer'])
l = len(new_answer)
em = exact_match_score(new_answer[:l], answer[:l])
print(em)

In [1]:
# musique bamboogle
import string
import re
import ast
def is_number_with_commas(s):
    pattern = re.compile(r'^\d{1,3}(,\d{3})*$')
    return bool(pattern.match(s))

def normalize_answer(s):
  def remove_articles(text):
    return re.sub(r"\b(a|an|the)\b", " ", text)
  
  def white_space_fix(text):
      return " ".join(text.split())

  def remove_punc(text):
      exclude = set(string.punctuation)
      return "".join(ch for ch in text if ch not in exclude)

  def lower(text):
      return text.lower()

  return white_space_fix(remove_articles(remove_punc(lower(s))))

def exact_match_score(predictions, references):
    """计算Exact Match得分"""
    cnt = 0
    for pred, ref in zip(predictions, references):
        # print(ref)
        gt = [normalize_answer(g) for g in ref]
        pred = [normalize_answer(p) for p in pred]
        get = False
        for p in pred:
            for g in gt:
                if(g in p):
                    cnt += 1
                    get = True
                    break
            if(get == True):
                break
    if len(predictions) == 0:
        return 0
    return cnt / len(predictions)

# 示例
import pandas as pd
import json

result_file = 'YOUR_PARSE_RESULT_FILE.csv' # your parse result file after running the parse_result.py

df1 = pd.read_csv(result_file)
answer = df1['Final Answer'].values.tolist()
new_answer = []
for idx in range(len(answer)):
    if(answer[idx]=='set()'):
        answer[idx] = '()'
    ans_set = ast.literal_eval(answer[idx])
    new_answer.append(list(ans_set))

raw_qa_json = '../fine-tuning/data/bamboogle.json' # '../fine-tuning/data/bamboogle.json' | '../fine-tuning/data/musique_500.json'
label = []
raw_qa = json.load(open(raw_qa_json,'r'))
for item in raw_qa:
    item['answer_2'].append(item['answer'])
    label.append(item['answer_2'])
l = len(new_answer)
em = exact_match_score(new_answer[:l], label[:l])
print(em)

In [4]:
# for CLQA_Wiki
from collections import Counter

import string
import re

def is_number_with_commas(s):
    pattern = re.compile(r'^\d{1,3}(,\d{3})*$')
    return bool(pattern.match(s))

def normalize_answer(s):
  def remove_articles(text):
    return re.sub(r"\b(a|an|the)\b", " ", text)
  
  def white_space_fix(text):
      return " ".join(text.split())

  def remove_punc(text):
      exclude = set(string.punctuation)
      return "".join(ch for ch in text if ch not in exclude)

  def lower(text):
      return text.lower()

  return white_space_fix(remove_articles(remove_punc(lower(s))))

def cal_score_single(prediction, reference):
    cnt = 0
    pred = [normalize_answer(s) for s in prediction]
    ref = [normalize_answer(s) for s in reference]
    for r in ref:
        if(r in pred): cnt += 1
    precision = 1.0 * cnt/len(prediction)
    recall = 1.0 * cnt/len(reference)

    return precision,recall

def cal_pre_recall(predictions,references):
    precision_scores = []
    recall_scores = []
    for i in range(len(predictions)):
        p,r = cal_score_single(predictions[i],references[i])
        precision_scores.append(p)
        recall_scores.append(r)
    return sum(precision_scores)/len(precision_scores),sum(recall_scores)/len(recall_scores)
    
import pandas as pd
import json
result_file = 'YOUR_PARSE_RESULT_FILE.csv' # your parse result file after running the parse_result.py
df1 = pd.read_csv(result_file)
answer = df1['Final Answer'].values.tolist()

new_answer = []
for idx in range(len(answer)):
    a = str(answer[idx])
    # print(a)
    a = a.strip("{}[]").strip("{}[]").strip("'")
    if is_number_with_commas(a):
        a_list = [a]
    else:
        a_list = a.split(",")
    for i in range(len(a_list)):
        a_list[i] = a_list[i].strip("'").strip().strip('"')
    new_answer.append(a_list)

label_file = '../CLQA_Wiki/CLQA_Wiki_v1.json'
label = []
q_type = []
with open(label_file,'r',encoding='utf-8') as f:
    data = json.load(f)
for d in data:
    label.append(d['label'])
    q_type.append(d['type'])
l = len(label)

result_2p = []
result_3p = []
result_2i = []
result_3i = []
result_ip = []
result_pi = []
result_2u = []
result_compare = []

all_p,all_r = 0,0

for i in range(l):
    ans = new_answer[i]
    t = q_type[i]
    p,r = cal_score_single(ans,label[i])
    all_p += p
    all_r += r
    if(t=='2p'):
        result_2p.append([p,r])
    if(t=='3p'):
        result_3p.append([p,r])
    if(t=='2i'):
        result_2i.append([p,r])
    if(t=='3i'):
        result_3i.append([p,r])
    if(t=='ip'):
        result_ip.append([p,r])
    if(t=='pi'):
        result_pi.append([p,r])
    if(t=='2u'):
        result_2u.append([p,r])
    if(t=='compare'):
        result_compare.append([p,r])
    
all_p_avg = all_p/l
all_r_avg = all_r/l

print(f"Avg Precision Score: {all_p_avg:.4f}; Avg Recall Score: {all_r_avg:.4f}")
